In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor

from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

from xgboost import XGBRegressor

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# File Path

file_path = r"D:\MSC THESIS AAVAS\AFTER MID DEFENSE\V85_Data_Excluding_9_Cooks_Distance_Points.xlsx"

# Load Dataset
df = pd.read_excel(file_path)

# Display First Five Rows
df.head()

In [ ]:
# Check Dataset Dimensions
print("Dataset Shape:", df.shape)

# Display Column Names
print("\nColumn Names:")
print(df.columns.tolist())

# Check Missing Values
print("\nMissing Values:")
print(df.isnull().sum())

# Display Descriptive Statistics
df.describe(include="all")

In [ ]:
df.describe()

In [ ]:
# Replace Infinity Radius with 1500 m

df["Radius of Curve"] = (
    df["Radius of Curve"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({
        "infinity": "1500",
        "inf": "1500",
        "∞": "1500"
    })
)

df["Radius of Curve"] = pd.to_numeric(
    df["Radius of Curve"],
    errors="raise"
)

# Verify
df["Radius of Curve"].describe()

In [ ]:
# Standardize Shoulder Type Categories

df["Left Shoulder Type"] = (
    df["Left Shoulder Type"]
    .str.strip()
    .str.lower()
    .str.capitalize()
)

df["Right Shoulder Type"] = (
    df["Right Shoulder Type"]
    .str.strip()
    .str.lower()
    .str.capitalize()
)

In [ ]:
# One-Hot Encoding

df_encoded = pd.get_dummies(
    df,
    columns=["Left Shoulder Type", "Right Shoulder Type"],
    drop_first=True,
    dtype=int
)

df_encoded.head()

In [ ]:
# Initial VIF Analysis

X_vif = df_encoded[
    [
        "Gradient",
        "Radius of Curve",
        "Carriageway Width",
        "Left Shoulder Width",
        "Right Shoulder Width",
        "Left Shoulder Type_Gravel",
        "Left Shoulder Type_Paved",
        "Right Shoulder Type_Gravel",
        "Right Shoulder Type_Paved",
        "IRI"
    ]
]

X_vif = X_vif.astype(float)

X_vif_const = sm.add_constant(X_vif)

vif_data = pd.DataFrame()

vif_data["Feature"] = X_vif_const.columns

vif_data["VIF"] = [
    variance_inflation_factor(X_vif_const.values, i)
    for i in range(X_vif_const.shape[1])
]

vif_data

In [ ]:
# Calculate Average Shoulder Width

df_encoded["Average Shoulder Width"] = (
    df_encoded["Left Shoulder Width"] +
    df_encoded["Right Shoulder Width"]
) / 2

# Display First Five Rows
df_encoded[
    [
        "Left Shoulder Width",
        "Right Shoulder Width",
        "Average Shoulder Width"
    ]
].head()

In [ ]:
# VIF with Average Shoulder Width

X_vif_avg = df_encoded[
    [
        "Gradient",
        "Radius of Curve",
        "Carriageway Width",
        "Average Shoulder Width",
        "Left Shoulder Type_Gravel",
        "Left Shoulder Type_Paved",
        "Right Shoulder Type_Gravel",
        "Right Shoulder Type_Paved",
        "IRI"
    ]
]

X_vif_avg = X_vif_avg.astype(float)

X_vif_avg_const = sm.add_constant(X_vif_avg)

vif_data_avg = pd.DataFrame()

vif_data_avg["Feature"] = X_vif_avg_const.columns

vif_data_avg["VIF"] = [
    variance_inflation_factor(X_vif_avg_const.values, i)
    for i in range(X_vif_avg_const.shape[1])
]

vif_data_avg

In [ ]:
# Final VIF Analysis Without Shoulder Types

X_vif_final = df_encoded[
    [
        "Gradient",
        "Radius of Curve",
        "Carriageway Width",
        "Average Shoulder Width",
        "IRI"
    ]
]

X_vif_final = X_vif_final.astype(float)

X_vif_final_const = sm.add_constant(X_vif_final)

vif_data_final = pd.DataFrame()

vif_data_final["Feature"] = X_vif_final_const.columns

vif_data_final["VIF"] = [
    variance_inflation_factor(X_vif_final_const.values, i)
    for i in range(X_vif_final_const.shape[1])
]

vif_data_final

In [ ]:
# Final Feature Selection

X = df_encoded[
    [
        "Gradient",
        "Radius of Curve",
        "Carriageway Width",
        "Average Shoulder Width",
        "IRI"
    ]
]

# Dependent Variable
y = df_encoded["V85 Speed"]

In [ ]:
print("Selected Independent Variables:")
print(X.columns.tolist())

print("\nNumber of Features:", X.shape[1])

print("Number of Observations:", X.shape[0])

print("\nTarget Variable:", y.name)

In [ ]:
# Distribution Plots

num_cols = [
    "V85 Speed",
    "Gradient",
    "Carriageway Width",
    "Average Shoulder Width",
    "Radius of Curve",
    "IRI"
]

sns.set_style("white")

for col in num_cols:
    plt.figure(figsize=(7, 4))

    sns.histplot(
        df_encoded[col],
        bins=20,
        stat="density",
        color="skyblue",
        edgecolor="black"
    )

    sns.kdeplot(
        df_encoded[col],
        color="red",
        linewidth=2
    )

    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Density")

    plt.tight_layout()
    plt.show()

In [ ]:
# Box Plots for Outlier Detection

sns.set_style("whitegrid")

for col in num_cols:
    plt.figure(figsize=(7, 4))

    sns.boxplot(
        x=df_encoded[col],
        color="lightblue",
        width=0.4
    )

    plt.title(f"Boxplot of {col}")
    plt.xlabel(col)

    plt.tight_layout()
    plt.show()

In [ ]:
# Skewness Analysis

skew_table = pd.DataFrame({
    "Variable": num_cols,
    "Skewness": df_encoded[num_cols].skew().values
})

skew_table

In [ ]:
# Pearson Correlation Matrix

corr_matrix = df_encoded[num_cols].corr()

corr_matrix

In [ ]:
# Correlation Heatmap

sns.set_style("white")

plt.figure(figsize=(9, 7))

sns.heatmap(
    corr_matrix,
    annot=True,
    cmap="RdYlGn",
    vmin=-1,
    vmax=1,
    center=0,
    fmt=".2f",
    linewidths=2,
    linecolor="black",
    square=True,
    annot_kws={"size": 11},
    cbar_kws={"shrink": 0.8}
)

plt.title(
    "Correlation Matrix of Variables",
    fontsize=15,
    fontweight="bold"
)

plt.xticks(rotation=45, ha="right", fontsize=11)
plt.yticks(rotation=0, fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# Regression Plots

predictor_cols = [
    "Radius of Curve",
    "Gradient",
    "Carriageway Width",
    "Average Shoulder Width",
    "IRI"
]

sns.set_style("white")

for col in predictor_cols:
    plt.figure(figsize=(7, 4))

    sns.regplot(
        x=df_encoded[col],
        y=df_encoded["V85 Speed"],
        scatter_kws={
            "s": 40,
            "color": "steelblue",
            "alpha": 0.7
        },
        line_kws={
            "color": "darkred",
            "linewidth": 2
        },
        ci=None
    )

    plt.title(f"{col} vs V85 Speed")
    plt.xlabel(col)
    plt.ylabel("V85 Speed")

    plt.tight_layout()
    plt.show()

In [ ]:
# Train-Test Split

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape :", y_test.shape)

In [ ]:
# Feature Scaling

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Multiple Linear Regression Model Training

from sklearn.linear_model import LinearRegression

mlr = LinearRegression()

mlr.fit(X_train, y_train)

mlr

In [ ]:
# Predictions Using MLR

y_train_pred_mlr = mlr.predict(X_train)

y_test_pred_mlr = mlr.predict(X_test)

In [ ]:
# MLR Model Evaluation Metrics

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

mlr_train_r2 = r2_score(y_train, y_train_pred_mlr)
mlr_test_r2 = r2_score(y_test, y_test_pred_mlr)

mlr_train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred_mlr))
mlr_test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred_mlr))

mlr_train_mae = mean_absolute_error(y_train, y_train_pred_mlr)
mlr_test_mae = mean_absolute_error(y_test, y_test_pred_mlr)

mlr_results = pd.DataFrame({
    "Metric": ["R2", "RMSE", "MAE"],
    "Value": [mlr_test_r2, mlr_test_rmse, mlr_test_mae]
})

mlr_results

In [ ]:
# MLR Training and Testing Performance

mlr_comparison = pd.DataFrame({
    "Metric": ["R2", "RMSE", "MAE"],
    "Training": [
        mlr_train_r2,
        mlr_train_rmse,
        mlr_train_mae
    ],
    "Testing": [
        mlr_test_r2,
        mlr_test_rmse,
        mlr_test_mae
    ]
})

mlr_comparison.round(4)

In [ ]:
# MLR Regression Coefficients

coef_table_mlr = pd.DataFrame({
    "Variable": X.columns,
    "Coefficient": mlr.coef_
})

# Add intercept as first row
intercept_row = pd.DataFrame({
    "Variable": ["Constant"],
    "Coefficient": [mlr.intercept_]
})

coef_table_mlr = pd.concat(
    [intercept_row, coef_table_mlr],
    ignore_index=True
)

coef_table_mlr["Coefficient"] = coef_table_mlr["Coefficient"].round(4)

coef_table_mlr

In [ ]:
# Random Forest Model Training

from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    random_state=42
)

# Train using unscaled data
rf.fit(X_train, y_train)

rf

In [ ]:
# Predictions Using Random Forest Model

y_train_pred_rf = rf.predict(X_train)

y_test_pred_rf = rf.predict(X_test)

In [ ]:
# Random Forest Model Evaluation

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np
import pandas as pd

# R2 Score
rf_train_r2 = r2_score(y_train, y_train_pred_rf)
rf_test_r2 = r2_score(y_test, y_test_pred_rf)

# RMSE
rf_train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred_rf))
rf_test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred_rf))

# MAE
rf_train_mae = mean_absolute_error(y_train, y_train_pred_rf)
rf_test_mae = mean_absolute_error(y_test, y_test_pred_rf)

# Model Evaluation Table
rf_results = pd.DataFrame({
    "Metric": ["R2", "RMSE", "MAE"],
    "Value": [rf_test_r2, rf_test_rmse, rf_test_mae]
})

rf_results

In [ ]:
# Random Forest Training and Testing Performance

rf_comparison = pd.DataFrame({
    "Metric": ["R2", "RMSE", "MAE"],
    "Training": [
        rf_train_r2,
        rf_train_rmse,
        rf_train_mae
    ],
    "Testing": [
        rf_test_r2,
        rf_test_rmse,
        rf_test_mae
    ]
})

rf_comparison.round(4)

In [ ]:
# Random Forest Feature Importance

rf_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
}).sort_values(
    by="Importance",
    ascending=False
)

rf_importance

In [ ]:
# Random Forest Feature Importance Plot

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

plt.figure(figsize=(7, 4))

sns.barplot(
    x="Importance",
    y="Feature",
    data=rf_importance,
    hue="Feature",
    palette="pastel",
    dodge=False,
    legend=False
)

plt.title(
    "Feature Importance (Random Forest)",
    fontsize=10,
    fontweight="bold"
)

plt.xlabel("Importance")
plt.ylabel("Feature")

plt.tight_layout()
plt.show()

In [ ]:
# Hyperparameter Grid for Random Forest

param_grid_rf = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 5, 10, 15],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

In [2]:
# Grid Search for Random Forest

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor

grid_rf = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=param_grid_rf,
    cv=5,
    scoring="r2",
    n_jobs=-1,
    verbose=1
)

# Train using unscaled data
grid_rf.fit(X_train, y_train)

print("Tuning completed")
print("Best Parameters:", grid_rf.best_params_)
print("Best CV Score:", round(grid_rf.best_score_, 3))

NameError: name 'param_grid_rf' is not defined

In [ ]:
# Extract Best Tuned Random Forest Model

best_rf = grid_rf.best_estimator_

best_rf

In [ ]:
# Predictions Using Tuned Random Forest

y_train_pred_rf_tuned = best_rf.predict(X_train)

y_test_pred_rf_tuned = best_rf.predict(X_test)

In [ ]:
# Tuned Random Forest Model Evaluation

# R2 Score
rf_tuned_train_r2 = r2_score(y_train, y_train_pred_rf_tuned)
rf_tuned_test_r2 = r2_score(y_test, y_test_pred_rf_tuned)

# RMSE
rf_tuned_train_rmse = np.sqrt(
    mean_squared_error(y_train, y_train_pred_rf_tuned)
)

rf_tuned_test_rmse = np.sqrt(
    mean_squared_error(y_test, y_test_pred_rf_tuned)
)

# MAE
rf_tuned_train_mae = mean_absolute_error(
    y_train, y_train_pred_rf_tuned
)

rf_tuned_test_mae = mean_absolute_error(
    y_test, y_test_pred_rf_tuned
)

# Model Evaluation Table
rf_tuned_results = pd.DataFrame({
    "Metric": ["R2", "RMSE", "MAE"],
    "Value": [
        rf_tuned_test_r2,
        rf_tuned_test_rmse,
        rf_tuned_test_mae
    ]
})

rf_tuned_results

In [ ]:
# Tuned Random Forest Training and Testing Performance

rf_tuned_comparison = pd.DataFrame({
    "Metric": ["R2", "RMSE", "MAE"],
    "Training": [
        rf_tuned_train_r2,
        rf_tuned_train_rmse,
        rf_tuned_train_mae
    ],
    "Testing": [
        rf_tuned_test_r2,
        rf_tuned_test_rmse,
        rf_tuned_test_mae
    ]
})

rf_tuned_comparison.round(4)

In [ ]:
# Tuned Random Forest Feature Importance

rf_tuned_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": best_rf.feature_importances_
}).sort_values(
    by="Importance",
    ascending=False
)

rf_tuned_importance

In [ ]:
# Tuned Random Forest Feature Importance Plot

sns.set_style("whitegrid")

plt.figure(figsize=(7, 4))

sns.barplot(
    x="Importance",
    y="Feature",
    data=rf_tuned_importance,
    hue="Feature",
    palette="pastel",
    dodge=False,
    legend=False
)

plt.title(
    "Feature Importance (Tuned Random Forest)",
    fontsize=10,
    fontweight="bold"
)

plt.xlabel("Importance")
plt.ylabel("Feature")

plt.tight_layout()
plt.show()